# [stand-in] CubeLang emitter — **gen 2** (v12e): the v8e recipe on gen 1 + the loop's own data

**What this is.** The same Unsloth LoRA SFT that produced `emitter_v8e` (LFM2.5-2.6B, r=32, 2 epochs, batch 32,
lr 2e-4 cosine, MAX_SEQ 4096, `<think>` trained empty), run on `emitter_sft_v12e.jsonl` — built by
`standin/data/build_gen2_partition.py` as **gen 1 unchanged + exactly two additions**, so any difference
between v8e and v12e on the gate is attributable:

1. **`chain` records from `cot_harvest_r7_v2.jsonl`** — chains the *emitter* planned, the grammar walked and the
   VM verified (exp_r7, 2026-09-11): the first training records that did not come from the grammar. They are
   included on the VM's word (`verified`), **never filtered by gold** — at serve time there is no gold and the
   loop's premise is that the VM is the truth gate; the manifest *reports* gold-correctness as a measurement of
   that gate. Upsampled ×6 (few records).
2. **A `plan` task** — prompt = the question alone (a plan is proposed before there are facts), target = a `CotPlan`
   binding `SEED` and `HOP1..k` relation strings in the question's own words, no objects:

   ```
   use vsa;
   program CotPlan implements ISolve {
       public function solve(mention: str): str {
           create frame: number;
           bind frame, SEED, "felix joseph widder";
           bind frame, HOP1, "country of citizenship";
           bind frame, HOP2, "administrative territorial entity";
           return recover(frame, SEED);
       }
   }
   ```

**Why gen 2 exists.** exp_r3 held v8e — trained only on the grammar's verified chains — to the 92 questions the grammar
misparses and the 37 it cannot parse, none of which were in its training data: gold hop count on 45/92, disposer-accepted
with gold hop count on **18/92** (the grammar's number is 0). Two of the three misparse forms were escaped; the third
(`3→2`, the `contained within the` joints) was inherited. Gen 2 asks whether the loop's own verified data, plus a task
whose target *is* the plan, moves that.

**The gate is pre-registered and excludes the training questions.** `build_gen2_partition.py` writes
`gen2_exclusions.json` (the r7 questions that entered training). Locally, after this notebook exports the GGUF:

```
python validation/exp_r3_emitter_floor.py --gguf standin/models/emitter_v12e.Q4_K_M.gguf \
    --exclude standin/data/out/gen2_exclusions.json --tag gen2
python validation/exp_r7_emitter_planned_walk.py --r3 validation/logs/exp_r3_emitter_floor_gen2.json \
    --exclude standin/data/out/gen2_exclusions.json --resident --tag _gen2
```

**Bar:** gen 2 escapes more of the *remaining* B questions than gen 1's 18/92 rate, at gen 1's precision or better,
same disposer, same VM. exp_r3 reads `CotPlan` output (the plan task) and `CotChain` output alike.

**Upload first** to `Drive/cubbyllm/standin/`: `emitter_sft_v12e.jsonl` + `emitter_sft_v12e.manifest.json`
(and `identity.py` if the repo checkout is unavailable, as before). Everything here is `[stand-in]`.


In [ ]:
# --- setup (run once per session) ---
import os, json, time, random, re
!pip -q install unsloth trl datasets
import sys
from google.colab import drive; drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/cubbyllm/standin'
if not os.path.exists('/content/CubbyLLM/.git'):
    !rm -rf /content/CubbyLLM && git clone https://github.com/Grillcheese-AI/CubbyLLM.git /content/CubbyLLM 2>&1 | tail -2
else:
    !cd /content/CubbyLLM && git fetch origin 2>&1 | tail -1 && git reset -q --hard origin/master && git log -1 --format='repo at %h %s'
sys.path.insert(0, '/content/CubbyLLM/standin/data'); sys.path.insert(0, '/content/CubbyLLM')
try:
    from identity import EMITTER_SYSTEM
    print('identity: from the repo checkout')
except ImportError as e:
    assert os.path.exists(f'{DRIVE}/identity.py'), f'identity.py not in the repo checkout nor at {DRIVE} ({e})'
    sys.modules.pop('identity', None); sys.path.insert(0, DRIVE)
    from identity import EMITTER_SYSTEM
    print('identity: from Drive')
VERSION = os.environ.get('STANDIN_VERSION', 'v12e')    # gen 2 of the program emitter; v8e was gen 1
DATA = f'{DRIVE}/emitter_sft_{VERSION}.jsonl'
MANIFEST = f'{DRIVE}/emitter_sft_{VERSION}.manifest.json'
OUT = f'{DRIVE}/emitter_lfm25_2p6b_{VERSION}'         # adapter + merged + GGUF land here
MODEL = os.environ.get('STANDIN_MODEL', 'LiquidAI/LFM2.5-2.6B')   # the v8e base; keep it, or the contrast is not attributable
MODEL_TAG = '' if MODEL == 'LiquidAI/LFM2.5-2.6B' else '_' + MODEL.split('/')[-1].lower().replace('.', 'p')
OUT = OUT + MODEL_TAG
EVAL_ONLY = os.environ.get('STANDIN_EVAL_ONLY') == '1'
if EVAL_ONLY: print('EVAL_ONLY: will load', f'{OUT}/merged', '(exists:', os.path.exists(f'{OUT}/merged'), ')')
MAX_SEQ = 4096   # the v8e setting (owner, 2026-09-03: 4096 trains better than 2048 on the A100-80G)
!nvidia-smi --query-gpu=name,memory.total --format=csv
for f in (DATA, MANIFEST):
    print(('ok      ' if os.path.exists(f) else 'MISSING ') + f)
m = json.load(open(MANIFEST))
assert 'r7_chains' in m and 'plan_records' in m, 'this is not a gen-2 manifest (build_gen2_partition.py writes r7_chains / plan_records)'
print('manifest', m['version'], ':', m['by_task'], '| records', m['n_records'], '| train rows after repeat', m['train_rows_after_repeat'])
print('gen 1 =', os.path.basename(m['gen1']), 'sha', m['gen1_sha256'][:12], '| r7 chains', m['r7_chains'], f"(x{m['r7_mult']})",
      '| r7 gold-correct', m['r7_gold_correct'], 'gold-wrong', m['r7_gold_wrong'], '(reported, not filtered)',
      '| plan records', m['plan_records'], '| excluded from the gate', m['n_excluded_questions'])


In [ ]:
# --- data: chat-format the records; train/val from the builder's deterministic split ---
SYSTEM = EMITTER_SYSTEM
recs = [json.loads(l) for l in open(DATA, encoding='utf-8')]
# the v8e filter (VM-ok, and not gold-wrong) -- EXCEPT the r7 chains, which enter on the VM's word alone:
# gold is not available at serve time, and filtering by it would make gen 2's data cleaner than the loop
# can ever produce. How many are gold-wrong is printed from the manifest above.
R7 = 'cubbyllm/cot_harvest_r7'
recs = [r for r in recs if r.get('vm_ok') in (True, None) and (r.get('gold_match') is not False or r.get('source') == R7)]
train = [r for r in recs if r['split'] == 'train']; val = [r for r in recs if r['split'] == 'val']
from collections import Counter
print('train', len(train), Counter(r['task'] for r in train)); print('val  ', len(val), Counter(r['task'] for r in val))
print('r7 chains in train:', sum(r.get('source') == R7 for r in train), '| plan records in train:', sum(r['task'] == 'plan' for r in train))

NO_THINK = '<think>\n</think>\n'   # LFM2.5 opens <think> on its own; train it to close immediately (v2, 2026-08-30)
def to_messages(r):
    return [{'role': 'system', 'content': r.get('system') or SYSTEM},
            {'role': 'user', 'content': r['prompt']},
            {'role': 'assistant', 'content': NO_THINK + r['program'].strip() + '\n'}]

from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained((f'{OUT}/merged' if EVAL_ONLY else MODEL), max_seq_length=MAX_SEQ, load_in_4bit=False, dtype=None)
print('model:', f'{OUT}/merged (trained, from Drive)' if EVAL_ONLY else MODEL)

def fmt(r):
    return {'text': tokenizer.apply_chat_template(to_messages(r), tokenize=False, add_generation_prompt=False)}
from datasets import Dataset
random.Random(0).shuffle(train)
ds_train = Dataset.from_list([fmt(r) for r in train for _ in range(int(r.get('repeat', 1)))])   # builder's `repeat`: chains x3, r7 x6
print('train rows after repeat weights:', len(ds_train))
lens = [len(tokenizer(x['text']).input_ids) for x in ds_train.select(range(min(500, len(ds_train))))]
print('token lengths (sample of 500): max', max(lens), 'p95', sorted(lens)[int(0.95*len(lens))], '-> MAX_SEQ', MAX_SEQ)
plan_example = next(r for r in train if r['task'] == 'plan')
print('\na plan record, formatted:\n', fmt(plan_example)['text'][:700])


In [ ]:
# --- LoRA + SFT: the v8e settings, unchanged ---
if EVAL_ONLY:
    print('EVAL_ONLY: skipping LoRA + training; the merged model from Drive is already loaded')
else:
    from trl import SFTTrainer, SFTConfig
    model = FastLanguageModel.get_peft_model(
        model, r=32, lora_alpha=32, lora_dropout=0.0, bias='none',
        target_modules=['q_proj', 'k_proj', 'v_proj', 'out_proj', 'o_proj', 'in_proj', 'w1', 'w2', 'w3', 'gate_proj', 'up_proj', 'down_proj'],
        use_gradient_checkpointing='unsloth', random_state=0)
    cfg = SFTConfig(output_dir='/content/emitter_ckpt', per_device_train_batch_size=32, gradient_accumulation_steps=1,
                    num_train_epochs=2, learning_rate=2e-4, lr_scheduler_type='cosine', warmup_steps=20,
                    logging_steps=10, save_strategy='no', bf16=True, max_seq_length=MAX_SEQ, dataset_text_field='text',
                    packing=False, report_to='none', seed=0)
    trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds_train, args=cfg)
    t0 = time.time(); stats = trainer.train(); print(f'trained in {(time.time()-t0)/60:.1f} min; final loss', stats.training_loss)
    os.makedirs(OUT, exist_ok=True); model.save_pretrained(f'{OUT}/adapter'); tokenizer.save_pretrained(f'{OUT}/adapter')
    json.dump({'version': VERSION, 'base': MODEL, 'data_sha256': m['output_sha256'], 'gen1_sha256': m['gen1_sha256'],
               'r7_sha256': m.get('r7_sha256'), 'train_rows': len(ds_train), 'final_loss': stats.training_loss,
               'lora': {'r': 32, 'alpha': 32, 'targets': 'q,k,v,out,o,in_proj,w1,w2,w3,gate,up,down'},
               'sft': {'epochs': 2, 'batch': 32, 'lr': 2e-4, 'sched': 'cosine', 'warmup': 20, 'max_seq': MAX_SEQ, 'seed': 0}},
              open(f'{OUT}/train_card.json', 'w'), indent=1)
    print('adapter ->', f'{OUT}/adapter', '| train card ->', f'{OUT}/train_card.json')


In [ ]:
# --- export FIRST: merged fp16 + GGUF (q8_0 for the VM eval, q4_k_m for the 12 GB Vulkan box) ---
if EVAL_ONLY:
    print('EVAL_ONLY: skipping export; the merged model from Drive is already loaded')
else:
    model.save_pretrained_merged(f'{OUT}/merged', tokenizer, save_method='merged_16bit')
    model.save_pretrained_gguf(f'{OUT}/gguf', tokenizer, quantization_method=['q8_0', 'q4_k_m'])
    import glob as _glob
    GGUFS = sorted(_glob.glob(f'{OUT}/gguf*/*.gguf'))
    print('GGUF files:'); [print('  ', g, f'{os.path.getsize(g)/1e9:.2f} GB') for g in GGUFS]
    print('\nlocally, as standin/models/emitter_v12e.Q4_K_M.gguf, the gate:')
    print('  python validation/exp_r3_emitter_floor.py --gguf standin/models/emitter_v12e.Q4_K_M.gguf --exclude standin/data/out/gen2_exclusions.json --tag gen2')
    print('  python validation/exp_r7_emitter_planned_walk.py --r3 validation/logs/exp_r3_emitter_floor_gen2.json --exclude standin/data/out/gen2_exclusions.json --resident --tag _gen2')


In [ ]:
# --- OPTIONAL format-level smoke eval on the val split (the verified read is local: eval_emitter_vm.py + exp_r3/exp_r7) ---
# `plan` is scored by exact match on the CotPlan text (deterministic given the question), and ALSO by
# "role chain matches" -- the relations and their order, ignoring the seed spelling -- which is the part the
# disposer reads. Set STANDIN_EVAL_N=0 to skip.
FastLanguageModel.for_inference(model)
import transformers; transformers.logging.set_verbosity_error()
def strip_think(s):
    return re.sub(r'^\s*(?:<think>)?.*?</think>\s*', '', s, count=1, flags=re.S) if '</think>' in s else s
def norm(s): return re.sub(r'\s+', ' ', re.sub(r'#.*', '', strip_think(s))).strip()
BINDS = re.compile(r'bind\s+frame\s*,\s*([A-Za-z_][A-Za-z0-9_]*)\s*,\s*"((?:[^"\\]|\\.)*)"\s*;')
def role_chain(prog):   # CotPlan: the HOPk strings in order; CotChain: the H-roles in order
    b = BINDS.findall(strip_think(prog))
    hops = sorted((int(k[3:]), v.lower()) for k, v in b if k.startswith('HOP') and k[3:].isdigit())
    if hops: return [v for _, v in hops]
    return [k for k, _ in b if re.match(r'^H\d+_', k)]
def emit(prompt, max_new=768, system=None):
    text = tokenizer.apply_chat_template([{'role': 'system', 'content': system or SYSTEM}, {'role': 'user', 'content': prompt}],
                                         tokenize=False, add_generation_prompt=True) + NO_THINK
    enc = tokenizer(text, return_tensors='pt', add_special_tokens=False).to('cuda')
    out = model.generate(**enc, max_new_tokens=max_new, do_sample=False, temperature=None, top_p=None,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
N_PER_TASK = int(os.environ.get('STANDIN_EVAL_N', '8'))
rng = random.Random(1); by_task = {}
for r in val: by_task.setdefault(r['task'], []).append(r)
sample = [r for t, rs in sorted(by_task.items()) for r in rng.sample(rs, min(N_PER_TASK, len(rs)))]
CAP = {'chain': 500, 'plan': 300, 'kernel': 600, 'arithmetic': 500, 'role_binding': 400}
print('eval sample', len(sample), {t: min(N_PER_TASK, len(rs)) for t, rs in sorted(by_task.items())})
hits = Counter(); tot = Counter(); chain_ok = Counter(); outputs = []
t0 = time.time()
for i, r in enumerate(sample):
    gen = emit(r['prompt'], max_new=CAP.get(r['task'], 600)); ok = norm(gen) == norm(r['program'])
    tot[r['task']] += 1; hits[r['task']] += int(ok)
    rc = None
    if r['task'] in ('plan', 'chain'):
        rc = role_chain(gen) == role_chain(r['program']); chain_ok[r['task']] += int(rc)
    outputs.append({'id': r['id'], 'task': r['task'], 'subtype': r.get('subtype', ''), 'source': r.get('source'),
                    'prompt': r['prompt'], 'reference': r['program'], 'generated': gen, 'exact_match': ok,
                    'role_chain_match': rc, 'gold': r.get('gold')})
    if (i+1) % 25 == 0: print(f'  {i+1}/{len(sample)} ({time.time()-t0:.0f}s)')
print('[stand-in] val exact-match by task:', {t: f'{hits[t]}/{tot[t]}' for t in tot}, '| overall', round(sum(hits.values())/max(1, sum(tot.values())), 3))
print('[stand-in] role-chain match (relations + order, what the disposer reads):', {t: f'{chain_ok[t]}/{tot[t]}' for t in chain_ok})
json.dump({'model': MODEL, 'version': VERSION, 'n': len(sample), 'exact_match_by_task': {t: hits[t]/tot[t] for t in tot},
           'role_chain_by_task': {t: chain_ok[t]/tot[t] for t in chain_ok}, 'outputs': outputs,
           'manifest_output_sha256': m['output_sha256']}, open(f'{OUT}/val_generations.json', 'w'), indent=1)
print('generations ->', f'{OUT}/val_generations.json')


### How to read

- **This notebook proves nothing about the gate.** Exact match on the val split is a format read; the `plan`
  role-chain match says the model learned the CotPlan shape and reproduces the *training* questions' chains.
  The gate is the two held-out arms on your machine, with the r7 questions excluded, scored by the disposer and
  walked through the VM — `exp_r3 --exclude` then `exp_r7 --exclude`.
- **What to compare, gen 1 → gen 2, on the same remaining questions:** disposer-accepted + gold hop count on
  arm B (gen 1: 18/92 before exclusion); VM-verified and gold-correct after the walk; and — the honest one —
  `verified_wrong`, which must not rise. If gen 2 escapes more but verifies wrong more, the loop is teaching
  confident wrong plans and the disposer, not the emitter, is what to look at next.
- **The `3→2` family** (`contained within the`, `office held by the head of`) had no verified example in gen 1's
  data. If r7 v2 delivered any, gen 2 is the first emitter that has seen one; report that family separately.
- **Everything here is `[stand-in]`.** It goes in `standin/README.md` and the TODO, never in
  `CUBBYLLM_HYPOTHESES.md` except as a pointer.
